In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agents-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Long-running agents — the core idea, worked

**A long-running agent is not a long-running process.** It wakes up, does *one* step, checkpoints, and goes back to sleep. The store is the only memory; a queue delivers wake-ups; waiting means *parking* the run until something calls `resume()`.

Five rules keep it safe — find the `## (n)` markers in `durable.py`:
1. **durable state** — every step ends in `save()`
2. **intent → act** — journal the call *with an idempotency key*, save, *then* do it; a retry repeats the same call and never re-asks the model
3. **lease** — one worker per run; leases *expire*
4. **budget** — a hard step limit in code
5. **park, don't wait** — a wait is a status + token, not a sleeping process

`durable.py` is ~170 lines. This notebook runs it.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))                      # repo root (run from notebooks/)
from durable import Agent, Crash, FakeClock, FakeModel, LeaseHeld, PaymentAPI, Queue, Store, Wait
RUNS = "runs.json"

def fresh(script, tools=None, clock=None):
    if os.path.exists(RUNS): os.remove(RUNS)
    pay = PaymentAPI()
    return Agent(Store(RUNS), Queue(), FakeModel(script), tools or {"charge": pay}, clock=clock or FakeClock()), pay

def show(run):
    print(f"{run.id}  status={run.status}  waiting_on={run.waiting_on}  result={run.result}")
    for i, s in enumerate(run.journal):
        print(f"  [{i}] {s['type']:<8}", {k: v for k, v in s.items() if k != "type"})

## 1. Happy path — one step per wake-up

In [2]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}])
run = agent.start("pay invoice 42")
while agent.queue.deliver_one(agent.handle):                 # each delivery = one HTTP request in prod
    print(f"wake-up #{agent.queue.delivered}: journal has {len(agent.store.get(run.id).journal)} entries, {len(agent.queue.items)} wake-up queued")
show(agent.store.get(run.id))
print("charges:", pay.charges)
print("on disk:", json.load(open(RUNS))[run.id]["status"])   # ← the only memory

wake-up #1: journal has 2 entries, 1 wake-up queued
wake-up #2: journal has 3 entries, 0 wake-up queued
run_487864  status=DONE  waiting_on=None  result=charged 42
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_487864:1', 'done': True, 'approved': False, 'result': {'charge_id': 'run_487864:1', 'amount': 42}}
  [2] decision {'final': 'charged 42'}
charges: {'run_487864:1': 42}
on disk: DONE


Read the journal: **decision → intent (with key `run:index`) → decision**. The intent line is what makes retries safe.

## 2. Crash *after* the card is charged, *before* the checkpoint
The worst window. We kill the process there, then let another worker retry.

In [3]:
clock = FakeClock()
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}], clock=clock)
run = agent.start("pay invoice 42")
agent.crash_at.add("after_side_effect")
try:
    agent.queue.deliver_one(agent.handle)
except Crash as e:
    print("💥", e)
show(agent.store.get(run.id))
print("charges:", pay.charges, "← money moved; journal says: intent, not done")

💥 process died after_side_effect
run_0da52f  status=RUNNING  waiting_on=None  result=None
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_0da52f:1', 'done': False, 'approved': False}
charges: {'run_0da52f:1': 42} ← money moved; journal says: intent, not done


In [4]:
worker2 = Agent(agent.store, agent.queue, agent.model, agent.tools, worker="worker-2", clock=clock)
try:
    agent.queue.deliver_one(worker2.handle)                  # the queue retries — too early
except LeaseHeld as e:
    print("retry refused:", e)
clock.advance(61)                                            # the dead worker's lease expires
agent.queue.drain(worker2.handle)                            # re-executes the SAME intent with the SAME key
show(agent.store.get(run.id))
print("charges:", pay.charges, "| model calls:", agent.model.calls, "← one charge, model never re-asked")

retry refused: run_0da52f is held by worker-1 until 1060
run_0da52f  status=DONE  waiting_on=None  result=charged 42
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_0da52f:1', 'done': True, 'approved': False, 'result': {'charge_id': 'run_0da52f:1', 'amount': 42}}
  [2] decision {'final': 'charged 42'}
charges: {'run_0da52f:1': 42} | model calls: 2 ← one charge, model never re-asked


Why not just re-ask the model on retry? Because it might answer differently (`amount=42.5`), producing a *second, different* charge. Journaling the decision turns a probabilistic step into a replayable one.

## 3. Duplicate delivery (queues are at-least-once)

In [5]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}])
run = agent.start("pay invoice 42")
agent.queue.duplicate_next()                                 # same message, delivered twice
agent.queue.drain(agent.handle)
print(agent.store.get(run.id).status, "| journal:", len(agent.store.get(run.id).journal), "| charges:", pay.charges, "| model calls:", agent.model.calls)

DONE | journal: 3 | charges: {'run_145514:1': 42} | model calls: 2


The wake-up carries `expect = (journal length, intent pending?)`; if the world has moved on, the delivery is a duplicate and does nothing.

## 4. Budget — the real stop condition

In [6]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 1}}] * 100)
run = agent.start("loop forever", max_steps=3)
agent.queue.drain(agent.handle)
print(agent.store.get(run.id).status, agent.store.get(run.id).result, "| charges:", len(pay.charges))

FAILED budget: 3 steps | charges: 3


## 5. Human gate — park with a token, execute exactly what was approved

In [7]:
pay = PaymentAPI(); pay.needs_approval = True
agent, _ = fresh([{"tool": "charge", "args": {"amount": 4200}}, {"final": "paid"}], tools={"charge": pay})
run = agent.start("pay invoice 4200")
agent.queue.drain(agent.handle)
run = agent.store.get(run.id)
print("parked:", run.status, run.waiting_on, "| queue:", agent.queue.items, "← nothing runs, nothing costs")
agent.resume(run.id, "wrong-token", {"approved": True}); print("wrong token →", agent.store.get(run.id).status)
agent.resume(run.id, run.waiting_on["token"], {"approved": True})
agent.resume(run.id, run.waiting_on["token"], {"approved": True})   # double click → no-op
agent.queue.drain(agent.handle)
show(agent.store.get(run.id)); print("charges:", pay.charges, "| model calls:", agent.model.calls)

parked: WAITING {'token': 'run_74c9d4:1', 'why': 'approval'} | queue: [] ← nothing runs, nothing costs
wrong token → WAITING
run_74c9d4  status=DONE  waiting_on=None  result=paid
  [0] decision {'tool': 'charge', 'args': {'amount': 4200}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 4200}, 'key': 'run_74c9d4:1', 'done': True, 'approved': True, 'result': {'charge_id': 'run_74c9d4:1', 'amount': 4200}}
  [2] decision {'final': 'paid'}
charges: {'run_74c9d4:1': 4200} | model calls: 2


## 6. Slow tool — the job runs elsewhere; the webhook resumes the run

In [8]:
def export(region, key):
    return Wait(token="job-" + key)                          # returns a handle, not a result
agent, _ = fresh([{"tool": "export", "args": {"region": "apac"}}, {"final": "report ready"}], tools={"export": export})
run = agent.start("export apac")
agent.queue.drain(agent.handle)
run = agent.store.get(run.id); print("parked:", run.status, run.waiting_on)
agent.resume(run.id, run.waiting_on["token"], {"rows": 1200})       # what the webhook (or a poller) does
agent.queue.drain(agent.handle)
show(agent.store.get(run.id))

parked: WAITING {'token': 'job-run_aa0ea7:1', 'why': 'event'}
run_aa0ea7  status=DONE  waiting_on=None  result=report ready
  [0] decision {'tool': 'export', 'args': {'region': 'apac'}}
  [1] intent   {'tool': 'export', 'args': {'region': 'apac'}, 'key': 'run_aa0ea7:1', 'done': True, 'approved': False, 'result': {'rows': 1200}}
  [2] decision {'final': 'report ready'}


Same primitive as the human gate: a status and a token. Who calls `resume()` differs — a person, a webhook, or a scheduler.

## 7. A real crash, across two processes
From a terminal in the repo root: `python demo.py kill` (charges, then `os._exit(137)`), then `python demo.py resume` — a fresh process finds the run in `runs.json` and finishes it with one charge.

## Where this goes on GCP
| Here | There |
|---|---|
| `Store` (JSON file) | Firestore, one document per run; `acquire_lease` = a transaction |
| `Queue` (named, at-least-once) | Cloud Tasks with named tasks, OIDC-authenticated HTTP target |
| `agent.handle(msg)` | `POST /internal/tasks/step` on Cloud Run (2xx only after a durable commit) |
| `resume()` | an HTTP endpoint hit by a person (IAP), a webhook, or Cloud Scheduler → Pub/Sub |
| `FakeModel` | Gemini on Vertex AI, given the journal as context |
| `PaymentAPI(key)` | any API with an `Idempotency-Key` header |
| `FakeClock.advance` | the lease TTL passing |

The full-blown version (Firestore/Cloud Tasks backends, fan-out/fan-in, sagas, scheduled agents, ADK 2 workflows, Terraform) is the `long-running-agents-gcp` repo — same five rules, more machinery.